# AI-Generated Content Detector — DistilBERT Fine-Tuning

Trains a DistilBERT model to classify text as **human-written** or **AI-generated**.

**Output**: HuggingFace-format model compatible with `AIGeneratedDetector` class.

**Target**: F1 >= 0.90

---

### Instructions
1. Set runtime to **GPU** (Runtime → Change runtime type → T4 GPU)
2. Run all cells
3. Download model, place in `backend/ml-services/nlp-service/models/ai-detector/`

In [ ]:
!pip install -q transformers datasets scikit-learn accelerate

In [ ]:
import os
import json
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from datasets import load_dataset

SEED = 42
MODEL_NAME = "distilbert-base-uncased"
OUTPUT_DIR = "/content/ai-detector"
EPOCHS = 5
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
MAX_LENGTH = 256

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

### Hugging Face login (optional)

Add `HF_TOKEN` in Colab Secrets (key icon → Secrets) to use gated/large datasets and higher rate limits. Then run the cell below.

In [ ]:
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Logged in to Hugging Face Hub.")
else:
    print("No HF_TOKEN set. Add in Colab Secrets to use gated datasets and higher rate limits.")

## 1. Load Dataset

Uses real AI-text detection datasets from HuggingFace.

In [ ]:
def _load_hc3_from_jsonl():
    """Load HC3 from raw JSONL (no dataset script; avoids deprecated trust_remote_code)."""
    base = "https://huggingface.co/datasets/Hello-SimpleAI/HC3/resolve/main"
    # Try single combined file first, then subset files
    urls = [f"{base}/all.jsonl", f"{base}/reddit_eli5.jsonl", f"{base}/finance.jsonl", f"{base}/medicine.jsonl"]
    for url in urls:
        try:
            ds = load_dataset("json", data_files=url, split="train")
            samples = []
            for row in ds:
                for ans in row.get("human_answers", []) or []:
                    s = str(ans).strip()
                    if len(s) > 20:
                        samples.append((s[:1000], 0))
                for ans in row.get("chatgpt_answers", []) or []:
                    s = str(ans).strip()
                    if len(s) > 20:
                        samples.append((s[:1000], 1))
            if samples:
                return samples, url.split("/")[-1]
        except Exception as e:
            continue
    return [], None


def load_ai_detection_data():
    """Load human (label 0) vs AI-generated (label 1) text. Ensures both classes exist."""
    all_samples = []

    # Try artem9k dataset first (best quality when it has both classes)
    try:
        print("Loading artem9k/ai-text-detection-pile...")
        ds = load_dataset("artem9k/ai-text-detection-pile", split="train")
        for row in ds:
            text = str(row.get("text", "")).strip()
            label = int(row.get("label", 0))
            if text and len(text) > 20:
                all_samples.append((text[:1000], label))
        n_h = sum(1 for _, l in all_samples if l == 0)
        n_a = sum(1 for _, l in all_samples if l == 1)
        print(f"  Loaded {len(all_samples)} samples (human: {n_h}, AI: {n_a})")
    except Exception as e:
        print(f"  Could not load: {e}")

    # Require both classes: load HC3 via raw JSONL (no deprecated script)
    n_human = sum(1 for _, l in all_samples if l == 0)
    n_ai = sum(1 for _, l in all_samples if l == 1)
    need_more = len(all_samples) < 1000 or n_human == 0 or n_ai == 0
    if need_more:
        print("Loading Hello-SimpleAI/HC3 from JSONL (for balance or extra data)...")
        hc3_samples, hc3_name = _load_hc3_from_jsonl()
        if hc3_samples:
            added_h = sum(1 for _, l in hc3_samples if l == 0)
            added_a = sum(1 for _, l in hc3_samples if l == 1)
            print(f"  HC3 ({hc3_name}): loaded {len(hc3_samples)} (human: {added_h}, AI: {added_a})")
            all_samples.extend(hc3_samples)
        else:
            print("  HC3 JSONL load failed (will try HC3-only below if needed).")
        n_human = sum(1 for _, l in all_samples if l == 0)
        n_ai = sum(1 for _, l in all_samples if l == 1)
        if n_ai == 0 or n_human == 0:
            # Still missing a class: use HC3-only
            print("  Using HC3-only (artem9k has no AI class)...")
            all_samples, hc3_name = _load_hc3_from_jsonl()
            if all_samples:
                n_h = sum(1 for _, l in all_samples if l == 0)
                n_a = sum(1 for _, l in all_samples if l == 1)
                print(f"  Loaded {len(all_samples)} from HC3 ({hc3_name}) (human: {n_h}, AI: {n_a})")
        n_human = sum(1 for _, l in all_samples if l == 0)
        n_ai = sum(1 for _, l in all_samples if l == 1)
        print(f"  Total samples: {len(all_samples)} (human: {n_human}, AI: {n_ai})")

    return all_samples


samples = load_ai_detection_data()
random.shuffle(samples)

# Balance so both classes are present (required for binary classification)
human = [(t, l) for t, l in samples if l == 0]
ai = [(t, l) for t, l in samples if l == 1]
min_count = min(len(human), len(ai))
if min_count == 0:
    raise ValueError(
        f"Both classes required but got human={len(human)}, AI={len(ai)}. "
        "artem9k may be human-only; ensure HC3 loaded (HF_TOKEN in Colab Secrets) or try another dataset."
    )
balanced = human[:min_count] + ai[:min_count]
random.shuffle(balanced)

texts = [s[0] for s in balanced]
labels = [s[1] for s in balanced]

if len(balanced) < 2:
    raise ValueError(
        f"Need at least 2 samples for train/val split; got {len(balanced)}. "
        "Check: (1) HuggingFace login (HF_TOKEN in Colab Secrets), (2) dataset access, (3) network."
    )

print(f"\nBalanced dataset: {len(texts)} samples")
print(f"  Human:        {len(texts) - sum(labels)}")
print(f"  AI-generated: {sum(labels)}")

## 2. Create DataLoaders

In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx], truncation=True, max_length=self.max_length,
            padding="max_length", return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
dataset = TextDataset(texts, labels, tokenizer, MAX_LENGTH)

n = len(dataset)
if n == 0:
    raise ValueError(
        "Dataset is empty. Run the data-loading cell above and ensure HuggingFace datasets load "
        "(e.g. set HF_TOKEN in Colab Secrets for gated datasets)."
    )

train_size = max(1, int(0.85 * n))
val_size = n - train_size
if val_size == 0:
    train_size -= 1
    val_size = 1
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

## 3. Train

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2,
    id2label={0: "human_written", 1: "ai_generated"},
    label2id={"human_written": 0, "ai_generated": 1},
)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=total_steps // 10, num_training_steps=total_steps
)

best_f1 = 0.0
best_metrics = {}

for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        batch_labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=batch_labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=-1)
        correct += (preds == batch_labels).sum().item()
        total += batch_labels.size(0)

    # Validation
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch["labels"].numpy())

    acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="binary")

    print(
        f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f} | "
        f"Train Acc: {correct/total:.4f} | Val Acc: {acc:.4f} | Val F1: {f1:.4f}"
    )

    if f1 > best_f1:
        best_f1 = f1
        best_metrics = {"accuracy": float(acc), "precision": float(prec), "recall": float(rec), "f1": float(f1)}
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)
        with open(os.path.join(OUTPUT_DIR, "training_metrics.json"), "w") as f:
            json.dump(best_metrics, f, indent=2)
        with open(os.path.join(OUTPUT_DIR, "model_card.json"), "w") as f:
            json.dump({
                "model_type": "ai-content-detector",
                "base_model": MODEL_NAME,
                "num_labels": 2,
                "labels": {"0": "human_written", "1": "ai_generated"},
                "metrics": best_metrics,
            }, f, indent=2)
        print(f"  -> Saved best model (F1: {best_f1:.4f})")

print(f"\nTraining complete! Best Val F1: {best_f1:.4f}")

## 4. Verify & Download

In [ ]:
# Quick inference test
test_model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR)
test_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
test_model.eval()

test_texts = [
    "I went to the store yesterday and bought some groceries. The weather was nice.",
    "As an AI language model, I can provide comprehensive information about various topics. Let me elaborate on the key aspects of this subject matter.",
    "Hey man, wanna grab lunch tomorrow? That new place on 5th looks good.",
    "In conclusion, it is imperative to consider the multifaceted implications of this approach. The synergistic integration of these methodologies yields optimal outcomes.",
]

for text in test_texts:
    inputs = test_tokenizer(text, truncation=True, max_length=256, padding=True, return_tensors="pt")
    with torch.no_grad():
        probs = torch.softmax(test_model(**inputs).logits, dim=-1)
    ai_prob = probs[0][1].item()
    label = "AI" if ai_prob > 0.5 else "HUMAN"
    print(f"[{label}] (p={ai_prob:.3f}) {text[:80]}...")

In [ ]:
# Download
!cd /content && zip -r ai-detector.zip ai-detector/
from google.colab import files
files.download("/content/ai-detector.zip")